# Ocean surface currents on the Spilhaus projection: three colour styles
**#30DayMapChallenge · Water**

The particles are simulated **once** and every frame is painted three ways, so you get three videos for roughly the cost of one:

- `spilhaus_cosmic.mp4`: the original (cmasher *cosmic*)
- `spilhaus_starry_night.mp4`: Van Gogh, *The Starry Night*
- `spilhaus_great_wave.mp4`: Hokusai, *The Great Wave off Kanagawa*

Colour = roughly how fast the water moves (gold / white = fastest).

## 1. Install (run once)

In [ ]:
%pip install -q pyproj xarray netCDF4 scipy matplotlib pillow "imageio[ffmpeg]" cmasher

## 2. Settings
Knobs: `GAIN` (brightness), `DT_HOURS` (particle speed), `FADE` (trail length), `N_PARTICLES` (density), `FPS` / `FRAMES_PER_MONTH` (video speed and length).

The three colour styles are in `STYLES`. Move a palette's gold/white stop (e.g. `0.90`) lower for more highlight colour, higher for less.

In [ ]:
import calendar
import numpy as np
import xarray as xr
import pyproj
import imageio.v2 as imageio
from scipy.interpolate import RegularGridInterpolator
from matplotlib import colormaps
import cmasher  # registers cmr.* colour maps
from PIL import Image, ImageDraw, ImageFont
from IPython.display import display, Video

print("PROJ", pyproj.proj_version_str, "(needs >= 9.6 for Spilhaus)")

IN_FILE = "currents/currents_2025_monthly.nc"

SIZE = 1080                 # output is SIZE x SIZE pixels
FPS = 45
FRAMES_PER_MONTH = 60       # 12 months x 60 = 720 frames = 16 s at 45 fps
N_PARTICLES = 120_000
DT_HOURS = 10               # simulated hours per frame (bigger = faster particles)
FADE = 0.95                 # trail persistence per frame (closer to 1 = longer trails)
LIFE = (60, 180)            # particle lifespan range in frames
COARSEN = 2                 # 0.083° -> ~0.17°; set 1 to keep full resolution
SPEED_REF = 0.6             # m/s that maps to the brightest colour
GAIN = 0.5                  # overall brightness of trails

# --- the three colour styles: output file -> colour map ---------------------------------
from matplotlib.colors import LinearSegmentedColormap

STYLES = {
    "spilhaus_cosmic.mp4": colormaps["cmr.cosmic"],

    # Van Gogh, The Starry Night: deep blues, chrome-yellow only on the fastest currents
    "spilhaus_starry_night.mp4": LinearSegmentedColormap.from_list("starry", [
        (0.00, "#0b1533"), (0.20, "#16306b"), (0.45, "#2756a8"), (0.65, "#4f86cf"),
        (0.80, "#7fa9cf"), (0.90, "#d9a521"), (1.00, "#f2c230")]),

    # Hokusai, The Great Wave: Prussian blue to foam-white on the fastest currents
    "spilhaus_great_wave.mp4": LinearSegmentedColormap.from_list("great_wave", [
        (0.00, "#0a1626"), (0.25, "#13304f"), (0.50, "#1f4e79"), (0.70, "#4f7fa6"),
        (0.85, "#9bb9cf"), (0.94, "#e6dfc8"), (1.00, "#f7f1e1")]),
}
LAND_RGB = np.array([28, 28, 32]) / 255
OUTSIDE_RGB = np.array([10, 10, 14]) / 255
R_EARTH = 6.371e6

## 3. Load the currents

In [ ]:
ds = xr.open_dataset(IN_FILE)
if "depth" in ds.dims:
    ds = ds.isel(depth=0, drop=True)
if COARSEN > 1:
    ds = ds.coarsen(latitude=COARSEN, longitude=COARSEN, boundary="trim").mean()
ds = ds.sortby("latitude").load()

lat = ds.latitude.values
lon = ds.longitude.values
times = ds.time.values
n_steps = len(times)
month_names = [calendar.month_name[int(str(t)[5:7])] for t in times]
print(f"grid {len(lat)} x {len(lon)}, {n_steps} time steps:", month_names)


def wrap_lon(a):
    """Pad both sides with the opposite edge column so interpolation is seamless at the dateline."""
    return np.concatenate([a[:, -1:], a, a[:, :1]], axis=1)


lon_w = np.concatenate([[lon[-1] - 360], lon, [lon[0] + 360]])
ocean = np.isfinite(ds.uo.isel(time=0).values)


def make_interp(arr):
    return RegularGridInterpolator((lat, lon_w), wrap_lon(np.nan_to_num(arr)),
                                   bounds_error=False, fill_value=0.0)


IU = [make_interp(ds.uo.isel(time=k).values) for k in range(n_steps)]
IV = [make_interp(ds.vo.isel(time=k).values) for k in range(n_steps)]
I_OCEAN = make_interp(ocean.astype(float))

## 4. Spilhaus projection, pixel grid and land mask
Every pixel is inverse-projected once to find what's inside the square and what's ocean.

In [ ]:
fwd = pyproj.Transformer.from_crs("EPSG:4326", "+proj=spilhaus", always_xy=True)

glon, glat = np.meshgrid(np.linspace(-180, 180, 1441), np.linspace(-90, 90, 721))
gx, gy = fwd.transform(glon, glat)
ok = np.isfinite(gx) & np.isfinite(gy)
xmin, xmax, ymin, ymax = gx[ok].min(), gx[ok].max(), gy[ok].min(), gy[ok].max()
span = max(xmax - xmin, ymax - ymin)
xmin, ymax = (xmin + xmax - span) / 2, (ymin + ymax + span) / 2  # square, centred
xmax, ymin = xmin + span, ymax - span

px = np.linspace(xmin, xmax, SIZE)
py = np.linspace(ymax, ymin, SIZE)
PX, PY = np.meshgrid(px, py)
plon, plat = fwd.transform(PX, PY, direction="INVERSE")
inside = np.isfinite(plon) & np.isfinite(plat) & (np.abs(plon) <= 180) & (np.abs(plat) <= 90)
is_ocean_px = np.zeros((SIZE, SIZE), bool)
is_ocean_px[inside] = I_OCEAN((plat[inside], (plon[inside] + 180) % 360 - 180)) > 0.5

background = np.empty((SIZE, SIZE, 3))
background[:] = OUTSIDE_RGB
background[inside & ~is_ocean_px] = LAND_RGB
ocean_px = inside & is_ocean_px


def to_pixels(lo, la):
    x, y = fwd.transform(lo, la)
    c = np.round((x - xmin) / span * (SIZE - 1))
    r = np.round((ymax - y) / span * (SIZE - 1))
    good = np.isfinite(c) & np.isfinite(r) & (c >= 0) & (c < SIZE) & (r >= 0) & (r < SIZE)
    return r[good].astype(int), c[good].astype(int), good


Image.fromarray((np.where(ocean_px[..., None], 0.35, background) * 255).astype(np.uint8)).resize((400, 400))

## 5. Particle simulation and colouring

In [ ]:
oc_lat_idx, oc_lon_idx = np.nonzero(ocean)
weights = np.cos(np.radians(lat[oc_lat_idx]))
weights /= weights.sum()
dlat = np.abs(np.diff(lat)).mean()
dlon = np.abs(np.diff(lon)).mean()

try:
    font = ImageFont.load_default(size=int(SIZE * 0.035))
    font_small = ImageFont.load_default(size=int(SIZE * 0.018))
except TypeError:  # Pillow < 10.1
    font = font_small = ImageFont.load_default()


def spawn(n, rng):
    """Spawn in ocean cells, weighted by cell area, jittered inside the cell."""
    k = rng.choice(len(weights), size=n, p=weights)
    la = lat[oc_lat_idx[k]] + rng.uniform(-0.5, 0.5, n) * dlat
    lo = lon[oc_lon_idx[k]] + rng.uniform(-0.5, 0.5, n) * dlon
    return (lo + 180) % 360 - 180, np.clip(la, -89.9, 89.9), rng.integers(*LIFE, n)


def simulate(frames_per_month=FRAMES_PER_MONTH, warmup=60, seed=42):
    """Run the particles once; yield (glow, t) per frame. glow is 0..1 per pixel."""
    rng = np.random.default_rng(seed)
    p_lon, p_lat, _ = spawn(N_PARTICLES, rng)
    p_age = rng.integers(0, LIFE[1], N_PARTICLES)
    dt = DT_HOURS * 3600
    trail = np.zeros((SIZE, SIZE), np.float32)
    n_frames = n_steps * frames_per_month

    for f in range(-warmup, n_frames):
        t = (f % n_frames) / frames_per_month
        i, w = int(t) % n_steps, t - int(t)
        j = (i + 1) % n_steps          # December eases back into January -> seamless loop

        pts = (p_lat, p_lon)
        u = (1 - w) * IU[i](pts) + w * IU[j](pts)
        v = (1 - w) * IV[i](pts) + w * IV[j](pts)
        speed = np.hypot(u, v)

        p_lon = (p_lon + np.degrees(u * dt / (R_EARTH * np.cos(np.radians(p_lat)))) + 180) % 360 - 180
        p_lat = np.clip(p_lat + np.degrees(v * dt / R_EARTH), -89.9, 89.9)

        p_age -= 1
        dead = (p_age <= 0) | (speed == 0)
        if dead.any():
            p_lon[dead], p_lat[dead], p_age[dead] = spawn(dead.sum(), rng)

        trail *= FADE
        r, c, good = to_pixels(p_lon, p_lat)
        np.add.at(trail, (r, c), 0.15 + speed[good] / SPEED_REF)

        if f >= 0:
            yield np.clip(1 - np.exp(-trail * GAIN), 0, 1), t


def colourise(glow, t, cmap):
    """Paint one simulated frame with a colour map, add labels; returns uint8 RGB."""
    img = background.copy()
    col = cmap(glow)[..., :3]
    img[ocean_px] = (OUTSIDE_RGB * 0.6 + col)[ocean_px].clip(0, 1)
    frame = Image.fromarray((img * 255).astype(np.uint8))
    d = ImageDraw.Draw(frame)
    d.text((SIZE * 0.035, SIZE * 0.03), month_names[int(round(t)) % n_steps],
           fill=(235, 235, 240), font=font)
    d.text((SIZE * 0.035, SIZE * 0.975),
           "Ocean surface currents · Spilhaus projection · Data: Copernicus Marine Service",
           fill=(150, 150, 160), font=font_small, anchor="ls")
    return np.asarray(frame)

## 6. Preview all three styles
Quick check before the full render. Tweak section 2, re-run it, then re-run this cell.

In [ ]:
gen = simulate(frames_per_month=6)
for _ in range(10):
    glow, t = next(gen)
previews = [Image.fromarray(colourise(glow, t, cmap)).resize((420, 420)) for cmap in STYLES.values()]
sheet = Image.new("RGB", (420 * len(previews) + 8 * (len(previews) - 1), 420), "white")
for k, p in enumerate(previews):
    sheet.paste(p, (k * 428, 0))
sheet

## 7. Render the three videos
One pass writes all three files. For a quick test clip set `frames_per_month=6`.

In [ ]:
writers = {name: imageio.get_writer(name, fps=FPS, quality=8, macro_block_size=1) for name in STYLES}
for k, (glow, t) in enumerate(simulate(frames_per_month=FRAMES_PER_MONTH)):
    for name, cmap in STYLES.items():
        writers[name].append_data(colourise(glow, t, cmap))
    if k % 60 == 0:
        print("frame", k, "/", n_steps * FRAMES_PER_MONTH)
for w in writers.values():
    w.close()
print("saved:", ", ".join(STYLES))

In [ ]:
for name in STYLES:
    print(name)
    display(Video(name, embed=True, width=480))

## 8. Optional: GIF versions

In [ ]:
GIF_SIZE = 480    # px
STEP = 4          # keep every 4th frame

for name in STYLES:
    reader = imageio.get_reader(name)
    gif_frames = [Image.fromarray(f).resize((GIF_SIZE, GIF_SIZE), Image.LANCZOS)
                  for k, f in enumerate(reader) if k % STEP == 0]
    gif_frames[0].save(name.replace(".mp4", ".gif"), save_all=True, append_images=gif_frames[1:],
                       duration=round(1000 * STEP / FPS), loop=0, optimize=True)
    print("saved", name.replace(".mp4", ".gif"))